In [6]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q transformers opencv-python-headless pandas torch torchvision ultralytics

import cv2, os, glob, torch, numpy as np, pandas as pd
from PIL import Image
from datetime import timedelta
from IPython.display import display
from transformers import CLIPProcessor, CLIPModel
from ultralytics import YOLO
from collections import Counter

print("Loading models...")
clip_model = CLIPModel.from_pretrained("openai/clip-vit-large-patch14")
clip_proc  = CLIPProcessor.from_pretrained("openai/clip-vit-large-patch14")
device     = "cuda" if torch.cuda.is_available() else "cpu"
clip_model = clip_model.to(device).eval()
yolo       = YOLO('yolov8s.pt')   # ← 's' model, better for low-res grayscale
print(f"Ready on {device} \n")

# ── Labels ───────────────────────────────────────────────────
CTX = "In a grayscale indoor surveillance camera footage, "
LABELS = [
    CTX + "a person is sitting on a chair or bench",
    CTX + "a person has fallen and is lying on the floor",
    CTX + "a person is walking slowly across the room",
    CTX + "a person is running or moving quickly",
    CTX + "two people are physically fighting or pushing each other",
    CTX + "a person is standing still not moving",
    CTX + "the room is empty with nobody in it",
]
LABEL_TO_EVENT = {
    CTX + "a person is sitting on a chair or bench"                  : "Person Sitting",
    CTX + "a person has fallen and is lying on the floor"            : "Person Collapsing",
    CTX + "a person is walking slowly across the room"               : "Person Walking",
    CTX + "a person is running or moving quickly"                    : "Person Running",
    CTX + "two people are physically fighting or pushing each other" : "Fighting",
    CTX + "a person is standing still not moving"                    : "Person Standing",
    CTX + "the room is empty with nobody in it"                      : "No Activity",
}

# ── Config ───────────────────────────────────────────────────
VIDEO_FOLDER  = '/content/drive/MyDrive/aiassignment3'
SAMPLE_EVERY  = 1      # classify every 1 second
SMOOTH_WINDOW = 3      # majority vote window to kill noise
MIN_SEG_SECS  = 3      # ignore segments shorter than 3s
SKIP_EVENTS   = {"No Activity"}

# ── Helpers ──────────────────────────────────────────────────
def fmt_timestamp(seconds):
    return str(timedelta(seconds=int(seconds))).zfill(8)   # 00:00:12

def fmt_frame_id(frame_num):
    return f"FRM_{frame_num:04d}"

def classify(frame_bgr):
    frame_resized = cv2.resize(frame_bgr, (336, 336))
    img    = Image.fromarray(cv2.cvtColor(frame_resized, cv2.COLOR_BGR2RGB))
    inputs = clip_proc(text=LABELS, images=img, return_tensors="pt", padding=True).to(device)
    with torch.no_grad():
        probs = clip_model(**inputs).logits_per_image.softmax(dim=-1)[0].cpu().numpy()
    idx = int(probs.argmax())
    return LABEL_TO_EVENT[LABELS[idx]], round(float(probs[idx]), 2)

def count_persons(frame_bgr):
    res = yolo(
        frame_bgr,
        verbose=False,
        conf=0.15,     # ← lowered from default 0.25 for grayscale footage
        iou=0.4,
        imgsz=640
    )
    return sum(1 for r in res[0].boxes if int(r.cls) == 0)

def get_videos(folder):
    files = []
    for ext in ['*.mpg','*.mpeg','*.mp4','*.avi','*.mov']:
        files.extend(glob.glob(os.path.join(folder, '**', ext), recursive=True))
        files.extend(glob.glob(os.path.join(folder, ext)))
    return list(set(files))

def smooth_labels(frame_labels, window=3):
    events   = [f[0] for f in frame_labels]
    smoothed = []
    half = window // 2
    for i in range(len(events)):
        lo  = max(0, i - half)
        hi  = min(len(events), i + half + 1)
        majority = Counter(events[lo:hi]).most_common(1)[0][0]
        smoothed.append(majority)
    return [(smoothed[i],) + frame_labels[i][1:] for i in range(len(frame_labels))]

def merge_into_segments(frame_labels):
    if not frame_labels:
        return []

    segments    = []
    cur_event   = frame_labels[0][0]
    cur_start_t = frame_labels[0][1]
    cur_start_f = frame_labels[0][2]
    cur_confs   = [frame_labels[0][3]]
    cur_persons = frame_labels[0][4]

    for (event, t_sec, frame_num, conf, persons) in frame_labels[1:]:
        if event == cur_event:
            cur_confs.append(conf)
            cur_persons = max(cur_persons, persons)
        else:
            segments.append({
                'event'     : cur_event,
                'start_sec' : cur_start_t,
                'frame_num' : cur_start_f,
                'conf'      : round(float(np.mean(cur_confs)), 2),
                'persons'   : cur_persons,
                'dur'       : t_sec - cur_start_t,
            })
            cur_event   = event
            cur_start_t = t_sec
            cur_start_f = frame_num
            cur_confs   = [conf]
            cur_persons = persons

    last = frame_labels[-1]
    segments.append({
        'event'     : cur_event,
        'start_sec' : cur_start_t,
        'frame_num' : cur_start_f,
        'conf'      : round(float(np.mean(cur_confs)), 2),
        'persons'   : cur_persons,
        'dur'       : last[1] - cur_start_t + SAMPLE_EVERY,
    })

    return [s for s in segments
            if s['dur'] >= MIN_SEG_SECS and s['event'] not in SKIP_EVENTS]

# ── Main ─────────────────────────────────────────────────────
def process_videos(folder):
    records     = []
    video_files = get_videos(folder)
    print(f"Found {len(video_files)} video(s)\n")

    for vpath in video_files:
        clip_id      = os.path.splitext(os.path.basename(vpath))[0]
        cap          = cv2.VideoCapture(vpath)
        fps          = cap.get(cv2.CAP_PROP_FPS) or 25
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        stride       = max(1, int(SAMPLE_EVERY * fps))

        print(f"▶ {clip_id}  ({total_frames/fps:.1f}s | {fps:.1f}fps)")

        raw_labels = []
        frame_num  = 0

        while frame_num < total_frames:
            cap.set(cv2.CAP_PROP_POS_FRAMES, frame_num)
            ret, frame = cap.read()
            if not ret:
                break

            event, conf = classify(frame)
            persons     = count_persons(frame)
            t_sec       = frame_num / fps
            raw_labels.append((event, t_sec, frame_num, conf, persons))
            frame_num  += stride

        cap.release()

        smoothed = smooth_labels(raw_labels, window=SMOOTH_WINDOW)
        segments = merge_into_segments(smoothed)

        print(f"  {'TIMESTAMP':<12} {'FRAME':<12} {'EVENT':<22} {'PERSONS':<18} CONF")
        print(f"  {'-'*72}")

        for s in segments:
            p_count = s['persons']

            # ── Fix: if CLIP sees activity but YOLO missed the person ──
            if p_count == 0 and s['event'] not in SKIP_EVENTS:
                p_str = "1 person "
            else:
                p_str = f"{p_count} person{'s' if p_count != 1 else ''}"

            ts  = fmt_timestamp(s['start_sec'])
            fid = fmt_frame_id(s['frame_num'])
            print(f"  {ts:<12} {fid:<12} {s['event']:<22} {p_str:<18} {s['conf']}")

            records.append({
                'Clip_ID'        : clip_id,
                'Timestamp'      : ts,
                'Frame_ID'       : fid,
                'Event_Detected' : s['event'],
                'Persons_Count'  : p_str,
                'Confidence'     : s['conf'],
            })

        print(f"\n   {len(segments)} activity blocks\n")

    return records

# ── Run ──────────────────────────────────────────────────────
print("=== CLIP CCTV Analysis (Final) ===\n")
records = process_videos(VIDEO_FOLDER)

if records:
    df = pd.DataFrame(records, columns=[
        'Clip_ID','Timestamp','Frame_ID',
        'Event_Detected','Persons_Count','Confidence'
    ])
    print(f"\n Final Table — {len(records)} events\n")
    display(df)
else:
    print("⚠️ No events found.")

print("\n💾 'df' ready in memory.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading models...


Loading weights:   0%|          | 0/590 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-large-patch14
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Ready on cuda 

=== CLIP CCTV Analysis (Final) ===

Found 5 video(s)

▶ Walk1  (24.7s | 25.0fps)
  TIMESTAMP    FRAME        EVENT                  PERSONS            CONF
  ------------------------------------------------------------------------
  00:00:03     FRM_0075     Person Running         1 person           0.32
  00:00:09     FRM_0225     Person Sitting         1 person           0.25
  00:00:12     FRM_0300     Person Running         1 person           0.35
  00:00:16     FRM_0400     Person Walking         1 person           0.26
  00:00:22     FRM_0550     Person Sitting         1 person           0.26

   5 activity blocks

▶ Browse2  (35.3s | 25.0fps)
  TIMESTAMP    FRAME        EVENT                  PERSONS            CONF
  ------------------------------------------------------------------------
  00:00:00     FRM_0000     Person Sitting         1 person           0.33
  00:00:04     FRM_0100     Person Running         1 person           0.42
  00:00:09     FRM_0225   

,Clip_ID,Timestamp,Frame_ID,Event_Detected,Persons_Count,Confidence
0,Walk1,00:00:03,FRM_0075,Person Running,1 person,0.32
1,Walk1,00:00:09,FRM_0225,Person Sitting,1 person,0.25
2,Walk1,00:00:12,FRM_0300,Person Running,1 person,0.35
3,Walk1,00:00:16,FRM_0400,Person Walking,1 person,0.26
4,Walk1,00:00:22,FRM_0550,Person Sitting,1 person,0.26
5,Browse2,00:00:00,FRM_0000,Person Sitting,1 person,0.33
6,Browse2,00:00:04,FRM_0100,Person Running,1 person,0.42
7,Browse2,00:00:09,FRM_0225,Person Sitting,1 person,0.30
8,Browse2,00:00:16,FRM_0400,Person Walking,1 person,0.38
9,Browse2,00:00:22,FRM_0550,Person Sitting,2 persons,0.66



💾 'df' ready in memory.


In [8]:
OUTPUT_CSV = '/content/drive/MyDrive/aiassignment3/student4_event_log.csv'
df.to_csv(OUTPUT_CSV, index=False)
print(f"✅ Saved: {OUTPUT_CSV}")

✅ Saved: /content/drive/MyDrive/aiassignment3/student4_event_log.csv
